In [17]:
import matplotlib.pyplot as plt


In [6]:
api_key = 'WxRSAcpuv90pVAdgPMmmQAAely5rpsJ6'

In [10]:
import pandas as pd
import numpy as np
import requests
from arch import arch_model
from scipy.stats import norm

# FMP API setup
# api_key = "YOUR_FMP_API_KEY"  # Replace with your FMP API key
base_url = "https://financialmodelingprep.com/api/v3/historical-price-full/SPY"
params = {
    "from": "2010-01-01",
    "to": "2024-12-31",
    "apikey": api_key
}

# Fetch SPY data from FMP
response = requests.get(base_url, params=params)
data = response.json()
prices = pd.DataFrame(data["historical"])
prices["date"] = pd.to_datetime(prices["date"])
prices = prices[["date", "adjClose"]].rename(columns={"adjClose": "Adj_Close"})
prices = prices.set_index("date").resample("W-FRI").last().dropna()  # Weekly data (Friday close)
prices["Returns"] = np.log(prices["Adj_Close"] / prices["Adj_Close"].shift(1))

# Initialize variables
cash = 0
shares = 0
portfolio_value = []
total_invested = 0
weeks = len(prices)
risk_free_rate = 0.02  # 2% annual
dividend_yield = 0.015  # 1.5% annual

# Black-Scholes call price function
def black_scholes_call(S, K, T, r, q, sigma):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

# Backtest loop
for week in range(weeks):
    # DCA: Invest $1,000
    spy_price = prices["Adj_Close"].iloc[week]
    shares_bought = 1000 / spy_price
    shares += shares_bought
    cash -= 1000
    total_invested += 1000

    # Sell covered call monthly (every 4 weeks)
    if week % 4 == 0 and week > 20:  # Start after enough data for GARCH
        # Fit GARCH(1,1) for volatility
        returns = prices["Returns"].iloc[:week].dropna() * 100  # Scale for GARCH
        model = arch_model(returns, vol="Garch", p=1, q=1, dist="Normal")
        fit = model.fit(disp="off")
        forecast = fit.forecast(horizon=21)  # 1-month forecast
        sigma = np.sqrt(forecast.variance.values[-1, :].mean()) / 100 * np.sqrt(252)  # Annualized volatility

        # Sell covered calls for half the shares
        num_contracts = int(shares // 200)  # 1 contract per 200 shares (half of holdings)
        if num_contracts > 0:
            K = spy_price * 1.05  # 5% OTM strike
            T = 1/12  # 1-month expiration
            premium = black_scholes_call(spy_price, K, T, risk_free_rate, dividend_yield, sigma)
            cash += num_contracts * premium * 100  # Premium per contract
            cash -= num_contracts * 1  # Transaction cost per contract

            # At expiration (next month)
            if week + 4 < weeks:
                spy_price_next = prices["Adj_Close"].iloc[week + 4]
                if spy_price_next > K:
                    # Option assigned: sell shares at strike
                    shares_sold = num_contracts * 100
                    shares -= shares_sold
                    cash += num_contracts * K * 100  # Proceeds from selling at strike
                    cash -= shares_sold * 0.01  # Transaction cost for shares
                # Else: Option expires worthless, keep premium

    # Calculate portfolio value
    portfolio_value.append(cash + shares * spy_price)

# Performance metrics
portfolio_value = np.array(portfolio_value)
returns = np.diff(portfolio_value) / portfolio_value[:-1]
annualized_return = (portfolio_value[-1] / total_invested) ** (1 / 5) - 1
volatility = np.std(returns) * np.sqrt(52)
sharpe_ratio = (annualized_return - risk_free_rate) / volatility
max_drawdown = np.max(np.maximum.accumulate(portfolio_value) - portfolio_value) / np.maximum.accumulate(portfolio_value)

# Output results
print(f"Final Portfolio Value: ${portfolio_value[-1]:,.2f}")
print(f"Total Invested: ${total_invested:,.2f}")
print(f"Annualized Return: {annualized_return:.2%}")
print(f"Volatility: {volatility:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown}")

Final Portfolio Value: $311,423.60
Total Invested: $783,000.00
Annualized Return: -16.84%
Volatility: 1856091627604027.25%
Sharpe Ratio: -0.00
Max Drawdown: [-3.63298243e+17 -3.63298243e+17 -3.63298243e+17 -3.63298243e+17
 -3.63298243e+17 -3.63298243e+17  5.78559556e+02  5.78559556e+02
  1.37408921e+02  1.03373375e+02  8.44962386e+01  7.52629943e+01
  6.06933596e+01  4.67347697e+01  4.67347697e+01  3.48201356e+01
  3.48201356e+01  3.48201356e+01  3.48201356e+01  3.48201356e+01
  3.48201356e+01  3.48201356e+01  3.48201356e+01  3.48201356e+01
  3.48201356e+01  3.48201356e+01  3.48201356e+01  3.48201356e+01
  3.48201356e+01  3.48201356e+01  3.48201356e+01  3.48201356e+01
  3.48201356e+01  3.48201356e+01  3.23678969e+01  2.91470448e+01
  1.99202799e+01  1.54341878e+01  1.54341878e+01  1.31025434e+01
  1.05510841e+01  1.00606675e+01  9.95592546e+00  7.56407539e+00
  7.08048905e+00  7.04455408e+00  7.04455408e+00  6.10536692e+00
  4.96232420e+00  4.86120020e+00  4.60208584e+00  4.57220981e+0

In [19]:
import pandas as pd
import numpy as np
import requests

# FMP API setup for SPY (SPDR S&P 500 ETF Trust, ticker: SPY)
base_url = "https://financialmodelingprep.com/api/v3/historical-price-full/SPY"
params = {
    "from": "2020-01-01",
    "to": "2024-12-31",
    "apikey": api_key
}

# Fetch SPY data from FMP
response = requests.get(base_url, params=params)
data = response.json()
prices = pd.DataFrame(data["historical"])
prices["date"] = pd.to_datetime(prices["date"])
prices = prices[["date", "adjClose"]].rename(columns={"adjClose": "Adj_Close"})
prices = prices.set_index("date").resample("W-FRI").last().dropna()  # Weekly data (Friday close)
prices = prices.resample("ME").first().dropna()  # Monthly data (first Friday of each month)

# Initialize variables
cash = 0  # Track cash for accounting, but not included in portfolio value
shares = 0
portfolio_value = []
total_invested = 0
months = len(prices)
risk_free_rate = 0.02  # 2% annual

# Backtest loop
for month in range(months):
    # DCA: Invest 10 times the minimum amount (10 * SPY_price, i.e., buy 10 shares)
    spy_price = prices["Adj_Close"].iloc[month]
    shares_bought = 10  # Buy exactly 10 shares
    investment_amount = 10 * spy_price  # Cost of 10 shares
    shares += shares_bought
    cash -= investment_amount
    total_invested += investment_amount  # Track total cash invested
    cash -= shares_bought * 0.01  # Transaction cost per share

    # Calculate portfolio value (exclude cash)
    portfolio_value.append(shares * spy_price)

# Performance metrics
portfolio_value = np.array(portfolio_value)
returns = np.diff(portfolio_value) / portfolio_value[:-1]
annualized_return = (portfolio_value[-1] / total_invested) ** (1 / 5) - 1
volatility = np.std(returns) * np.sqrt(12)  # Annualized volatility (monthly returns)
sharpe_ratio = (annualized_return - risk_free_rate) / volatility
# Max drawdown calculation
running_max = np.maximum.accumulate(portfolio_value)
drawdowns = (running_max - portfolio_value) / running_max
max_drawdown = np.max(drawdowns).item()  # Convert to scalar

# Output results
print(f"Final Portfolio Value: ${portfolio_value[-1]:,.2f}")
print(f"Total Invested: ${total_invested:,.2f}")
print(f"Annualized Return: {annualized_return:.2%}")
print(f"Volatility: {volatility:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")

Final Portfolio Value: $356,435.20
Total Invested: $250,607.10
Annualized Return: 7.30%
Volatility: 53.42%
Sharpe Ratio: 0.10
Max Drawdown: 6.40%


In [21]:
import pandas as pd
import numpy as np
import requests
from arch import arch_model
from scipy.stats import norm

# FMP API setup for SPY (SPDR S&P 500 ETF Trust, ticker: SPY)
base_url = "https://financialmodelingprep.com/api/v3/historical-price-full/SPY"
params = {
    "from": "2020-01-01",
    "to": "2024-12-31",
    "apikey": api_key
}

# Fetch SPY data from FMP
response = requests.get(base_url, params=params)
data = response.json()
prices = pd.DataFrame(data["historical"])
prices["date"] = pd.to_datetime(prices["date"])
prices = prices[["date", "adjClose"]].rename(columns={"adjClose": "Adj_Close"})
prices = prices.set_index("date").resample("W-FRI").last().dropna()  # Weekly data (Friday close)
prices = prices.resample("ME").first().dropna()  # Monthly data (first Friday of each month)
prices["Returns"] = np.log(prices["Adj_Close"] / prices["Adj_Close"].shift(1))

# Initialize variables
cash = 0  # Track cash for portfolio value
shares = 0
portfolio_value = []
total_invested = 0
months = len(prices)
risk_free_rate = 0.02  # 2% annual
dividend_yield = 0.015  # 1.5% annual

# Black-Scholes call price function
def black_scholes_call(S, K, T, r, q, sigma):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

# Backtest loop
for month in range(months):
    # DCA: Invest 10 times the minimum amount (10 * SPY_price, i.e., buy 10 shares)
    spy_price = prices["Adj_Close"].iloc[month]
    shares_bought = 10  # Buy exactly 10 shares
    investment_amount = 10 * spy_price  # Cost of 10 shares
    shares += shares_bought
    cash -= investment_amount
    total_invested += investment_amount  # Track total cash invested
    cash -= shares_bought * 0.01  # Transaction cost per share

    # Sell covered call (on half the shares)
    if month > 5:  # Start after enough data for GARCH
        # Fit GARCH(1,1) for volatility
        returns = prices["Returns"].iloc[:month].dropna() * 100  # Scale for GARCH
        model = arch_model(returns, vol="Garch", p=1, q=1, dist="Normal")
        fit = model.fit(disp="off")
        forecast = fit.forecast(horizon=21)  # 1-month forecast
        sigma = np.sqrt(forecast.variance.values[-1, :].mean()) / 100 * np.sqrt(252)  # Annualized volatility

        # Sell covered calls for half the shares
        num_contracts = int(shares // 200)  # 1 contract per 200 shares (half of holdings)
        if num_contracts > 0:
            K = spy_price * 1.05  # 5% OTM strike
            T = 1/12  # 1-month expiration
            premium = black_scholes_call(spy_price, K, T, risk_free_rate, dividend_yield, sigma)
            cash += num_contracts * premium * 100  # Premium per contract
            cash -= num_contracts * 1  # Transaction cost per contract

            # At expiration (next month)
            if month + 1 < months:
                spy_price_next = prices["Adj_Close"].iloc[month + 1]
                if spy_price_next > K:
                    # Option assigned: sell shares at strike
                    shares_sold = num_contracts * 100
                    shares -= shares_sold
                    cash += num_contracts * K * 100  # Proceeds from selling at strike
                    cash -= shares_sold * 0.01  # Transaction cost for shares
                # Else: Option expires worthless, keep premium

    # Calculate portfolio value (include cash)
    portfolio_value.append(cash + shares * spy_price)

# Performance metrics
portfolio_value = np.array(portfolio_value)
returns = np.diff(portfolio_value) / portfolio_value[:-1]
annualized_return = (portfolio_value[-1] / total_invested) ** (1 / 5) - 1
volatility = np.std(returns) * np.sqrt(12)  # Annualized volatility (monthly returns)
sharpe_ratio = (annualized_return - risk_free_rate) / volatility
# Max drawdown calculation
running_max = np.maximum.accumulate(portfolio_value)
drawdowns = (running_max - portfolio_value) / running_max
max_drawdown = np.max(drawdowns).item()  # Convert to scalar

# Output results
print(f"Final Portfolio Value: ${portfolio_value[-1]:,.2f}")
print(f"Total Invested: ${total_invested:,.2f}")
print(f"Annualized Return: {annualized_return:.2%}")
print(f"Volatility: {volatility:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")

Final Portfolio Value: $123,419.74
Total Invested: $250,607.10
Annualized Return: -13.21%
Volatility: 40045.84%
Sharpe Ratio: -0.00
Max Drawdown: 2181.93%
